# Task 3: Data Cleaning and Text Preprocessing
Goal: clean the TED Talks dataset and build a combined_text column for text vectorization (next phase).
The original file in data/raw/ is never modified.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW_PATH = Path("../data/raw/ted_talks.csv")
PROCESSED_PATH = Path("../data/processed/ted_talks_cleaned.csv")
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)


In [2]:
raw_df = pd.read_csv(RAW_PATH)
df = raw_df.copy()      # we only edit df; raw_df stays untouched
print("Shape:", raw_df.shape)
raw_df.head(3)

Shape: (4005, 19)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."


In [3]:
before = {
    "rows": len(raw_df),
    "columns": raw_df.shape[1],
    "exact_duplicate_rows": int(raw_df.duplicated().sum()),
    "duplicate_talk_id": int(raw_df["talk_id"].duplicated().sum()),
    "duplicate_url": int(raw_df["url"].duplicated().sum()),
}
before

{'rows': 4005,
 'columns': 19,
 'exact_duplicate_rows': 0,
 'duplicate_talk_id': 0,
 'duplicate_url': 0}

In [4]:
missing = raw_df.isna().sum()
blank = raw_df.select_dtypes(include="object").apply(lambda s: (s.str.strip() == "").sum())
summary = pd.DataFrame({"missing (NaN)": missing, "blank strings": blank}).fillna(0).astype(int)
summary[(summary > 0).any(axis=1)]

C:\Users\oop\AppData\Local\Temp\ipykernel_15896\460181126.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  blank = raw_df.select_dtypes(include="object").apply(lambda s: (s.str.strip() == "").sum())


,missing (NaN),blank strings
about_speakers,503,0
all_speakers,4,0
comments,655,0
occupations,522,0
recorded_date,1,0


## Review of Task 2 findings
- Dataset: 4005 rows, 19 columns.
- Missing values: comments (655), occupations (522), about_speakers (503), all_speakers (4), recorded_date (1).
- No missing values in title, description, topics, transcript, url.
- Fields needing cleaning: title, topics, description, transcript (used for recommendations).
- The original file in data/raw/ is preserved. All work is done on a copy.

## Missing-value decisions

| Column | Missing | Decision | Reason |
|---|---|---|---|
| all_speakers | 4 | Fill from speaker_1 | Same kind of info, so no gap |
| occupations | 522 | Fill with empty string | Keep the talk; no fake word added |
| about_speakers | 503 | Fill with empty string | Keep the talk; no fake word added |
| comments | 655 | Leave as is | Number, not used for text matching |
| recorded_date | 1 | Leave as is | Only 1 row, not used for recommendations |
| title, description, topics, transcript, url | 0 | No action | Nothing missing |

Rows are not deleted because every talk still has a title, topics, description and transcript.

In [5]:
df["all_speakers"] = df["all_speakers"].fillna(df["speaker_1"])

text_cols = ["title", "topics", "description", "transcript",
             "occupations", "about_speakers", "all_speakers"]
for col in text_cols:
    df[col] = df[col].fillna("").astype(str).str.strip()   # missing and blank both become ""

In [6]:
after_missing = pd.DataFrame({"before": raw_df.isna().sum(), "after": df.isna().sum()})
after_missing[(after_missing > 0).any(axis=1)]

,before,after
all_speakers,4,0
occupations,522,0
about_speakers,503,0
recorded_date,1,1
comments,655,655


In [7]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated talk_id:", df["talk_id"].duplicated().sum())
print("Repeated url:", df["url"].duplicated().sum())
print("Repeated title (lowercase, trimmed):", df["title"].str.lower().str.strip().duplicated().sum())
print("Repeated url (lowercase, no trailing slash):",
      df["url"].str.lower().str.strip().str.rstrip("/").duplicated().sum())

Exact duplicate rows: 0
Repeated talk_id: 0
Repeated url: 0
Repeated title (lowercase, trimmed): 0
Repeated url (lowercase, no trailing slash): 0


In [8]:
dup_ids = df[df.duplicated("talk_id", keep=False)].sort_values("talk_id")
dup_urls = df[df.duplicated("url", keep=False)].sort_values("url")
print("Rows sharing a talk_id:", len(dup_ids))
print("Rows sharing a url:", len(dup_urls))
dup_ids[["talk_id", "title", "speaker_1", "url"]].head(10)

Rows sharing a talk_id: 0
Rows sharing a url: 0


,talk_id,title,speaker_1,url


In [9]:
rows_before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="talk_id")
df = df.drop_duplicates(subset="url")
df = df.reset_index(drop=True)
print("Rows before:", rows_before)
print("Rows after:", len(df))
print("Duplicates removed:", rows_before - len(df))

Rows before: 4005
Rows after: 4005
Duplicates removed: 0


Checked exact rows, talk_id, url and cleaned title. Duplicates found: 0. Removed: 0.


## Cleaning rules
1. Lowercase
2. Remove HTML tags, links and audience cues like (Laughter) and (Applause)
3. Punctuation becomes a space; apostrophes inside words are kept (don't stays don't)
4. Repeated spaces and leading/trailing spaces are removed
5. Missing and blank text are both ""
6. Original columns are kept; cleaned versions go in new *_clean columns
7. Stop-words are NOT removed here; the vectorizer will handle them in the next phase

In [10]:
CUES = r"\((?:laughter|applause|music|video|audio|singing|cheering|cheers|laughs|applause ends)[^)]*\)"

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace("’", "'").replace("‘", "'")        # curly to straight apostrophes
    text = re.sub(r"<[^>]+>", " ", text)                    # html tags
    text = re.sub(CUES, " ", text)                          # (laughter), (applause)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)      # links
    text = re.sub(r"[^\w\s']", " ", text)                   # punctuation -> space
    text = text.replace("_", " ")
    text = re.sub(r"(?<!\w)'|'(?!\w)", " ", text)           # apostrophes not inside a word
    text = re.sub(r"\s+", " ", text).strip()                # extra spaces
    return text

In [11]:
for col in ["title", "topics", "description", "transcript"]:
    df[col + "_clean"] = df[col].apply(clean_text)
df[["title_clean", "topics_clean", "description_clean", "transcript_clean"]].head(3)

,title_clean,topics_clean,description_clean,transcript_clean
0,averting the climate crisis,alternative energy cars climate change culture...,with the same humor and humanity he exuded in ...,thank you so much chris and it's truly a great...
1,the best stats you've ever seen,africa asia google demo economics global issue...,you've never seen data presented like this wit...,about 10 years ago i took on the task to teach...
2,simplicity sells,computers entertainment interface design media...,new york times columnist david pogue takes aim...,hello voice mail my old friend i've called for...


In [12]:
pd.set_option("display.max_colwidth", 120)
df[["title", "title_clean", "topics", "topics_clean"]].sample(3, random_state=1)

,title,title_clean,topics,topics_clean
870,The antidote to apathy,the antidote to apathy,"['TEDx', 'collaboration', 'community', 'culture', 'global issues', 'politics']",tedx collaboration community culture global issues politics
929,Try something new for 30 days,try something new for 30 days,"['culture', 'success']",culture success
1669,Be passionate. Be courageous. Be your best.,be passionate be courageous be your best,"['violence', 'guns']",violence guns


In [13]:
for i in df.sample(2, random_state=7).index:
    print("DESCRIPTION ORIGINAL:", df.loc[i, "description"][:250])
    print("DESCRIPTION CLEANED :", df.loc[i, "description_clean"][:250])
    print("TRANSCRIPT ORIGINAL :", df.loc[i, "transcript"][:250])
    print("TRANSCRIPT CLEANED  :", df.loc[i, "transcript_clean"][:250])
    print("-" * 80)

DESCRIPTION ORIGINAL: "Life comes at us very quickly, and what we need to do is take that amorphous flow of experience and somehow extract meaning from it." In this funny, enlightening talk, educational psychologist Peter Doolittle details the importance -- and limitation
DESCRIPTION CLEANED : life comes at us very quickly and what we need to do is take that amorphous flow of experience and somehow extract meaning from it in this funny enlightening talk educational psychologist peter doolittle details the importance and limitations of your
TRANSCRIPT ORIGINAL : So yesterday, I was out in the street in front of this building, and I was walking down the sidewalk, and I had company, several of us, and we were all abiding by the rules of walking down sidewalks. We're not talking each other. We're facing forward
TRANSCRIPT CLEANED  : so yesterday i was out in the street in front of this building and i was walking down the sidewalk and i had company several of us and we were all abiding by t

In [14]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

sample = df.loc[0, "description_clean"]
tokens = sample.split()
tokens_no_stop = [w for w in tokens if w not in ENGLISH_STOP_WORDS]
print("Tokens:", tokens[:15])
print("Without stop-words:", tokens_no_stop[:15])
print("Count:", len(tokens), "->", len(tokens_no_stop))

Tokens: ['with', 'the', 'same', 'humor', 'and', 'humanity', 'he', 'exuded', 'in', 'an', 'inconvenient', 'truth', 'al', 'gore', 'spells']
Without stop-words: ['humor', 'humanity', 'exuded', 'inconvenient', 'truth', 'al', 'gore', 'spells', '15', 'ways', 'individuals', 'address', 'climate', 'change', 'immediately']
Count: 39 -> 23


In [15]:
df["combined_text"] = (
    df["title_clean"] + " " + df["topics_clean"] + " " +
    df["description_clean"] + " " + df["transcript_clean"]
).str.replace(r"\s+", " ", regex=True).str.strip()

print("Empty combined_text rows:", (df["combined_text"] == "").sum())
print(df["combined_text"].str.split().str.len().describe())
df[["title", "combined_text"]].assign(combined_text=lambda d: d["combined_text"].str[:200]).head(3)

Empty combined_text rows: 0
count     4005.000000
mean      1866.370037
std        998.646226
min         36.000000
25%        991.000000
50%       1837.000000
75%       2516.000000
max      11216.000000
Name: combined_text, dtype: float64


,title,combined_text
0,Averting the climate crisis,averting the climate crisis alternative energy cars climate change culture environment global issues science sustain...
1,The best stats you've ever seen,the best stats you've ever seen africa asia google demo economics global issues health statistics global development...
2,Simplicity sells,simplicity sells computers entertainment interface design media music performance simplicity software technology new...


In [16]:
save_cols = list(raw_df.columns) + ["title_clean", "topics_clean", "description_clean", "combined_text"]
df[save_cols].to_csv(PROCESSED_PATH, index=False, encoding="utf-8")

check = pd.read_csv(PROCESSED_PATH)
print("Saved shape:", check.shape)
print("combined_text empty/NaN:", check["combined_text"].isna().sum())
print("File size (MB):", round(PROCESSED_PATH.stat().st_size / 1024 / 1024, 1))
print("Raw file unchanged:", pd.read_csv(RAW_PATH).equals(raw_df))

Saved shape: (4005, 23)
combined_text empty/NaN: 0
File size (MB): 83.9
Raw file unchanged: True


In [17]:
comparison = pd.DataFrame({
    "Before": [before["rows"], before["columns"], before["exact_duplicate_rows"],
               before["duplicate_talk_id"], before["duplicate_url"], int(raw_df.isna().sum().sum())],
    "After":  [len(df), len(save_cols), int(df.duplicated().sum()),
               int(df["talk_id"].duplicated().sum()), int(df["url"].duplicated().sum()),
               int(df[list(raw_df.columns)].isna().sum().sum())]
}, index=["Rows", "Columns (After = saved file)", "Exact duplicate rows", "Repeated talk_id", "Repeated url", "Total missing cells"])
comparison

,Before,After
Rows,4005,4005
Columns (After = saved file),19,23
Exact duplicate rows,0,0
Repeated talk_id,0,0
Repeated url,0,0
Total missing cells,1685,656


## Short report

**Steps done**
1. Loaded ted_talks.csv from data/raw/ and worked on a copy; the original is untouched.
2. Missing values: filled all_speakers from speaker_1; filled occupations and about_speakers with empty strings; left comments and recorded_date as they are. No talks were removed.
3. Duplicates: checked exact rows, talk_id, url and cleaned title. Found __ , removed __ . Rows: 4005 before, ____ after.
4. Text cleaning on title, topics, description, transcript: lowercase, removed HTML/links/audience cues, punctuation to spaces (apostrophes inside words kept), extra spaces removed. Originals kept; cleaned versions in *_clean columns.
5. Built combined_text = title + topics + description + transcript.
6. Saved to data/processed/ted_talks_cleaned.csv.

**Limitations**
- Transcripts are much longer than titles and topics, so they will dominate similarity scores. Next phase can weight fields.
- Lowercasing loses case clues (e.g. AI vs ai).
- Possessives like dogs' lose the apostrophe.
- Removing audience cues uses a fixed list; unusual cues may remain.
- Stop-words and stemming are not applied yet; they are left to the vectorizer.
- Speaker fields (occupations, about_speakers) are cleaned of gaps but not included in combined_text.
- Empty strings become NaN when the CSV is read again; fill them before vectorizing.